# HCP1065 PyAFQ centroid QC (3D)

Loads centroid `.npy` arrays from:
`data/atlases/HCP1065/centroids`

Each file contains a `(100, 3)` array of MNI coordinates (one 3D point per node along the tract).

In [5]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import ipywidgets as widgets
from IPython.display import display

BASE_DIR = Path("__PROJECT_ROOT__")
CENTROIDS_DIR = BASE_DIR / "data" / "atlases" / "HCP1065" / "centroids"

assert CENTROIDS_DIR.exists(), f"Missing centroids dir: {CENTROIDS_DIR}"

centroid_files = sorted(CENTROIDS_DIR.glob("*_model_centroids.npy"))
len(centroid_files), (centroid_files[0].name if centroid_files else None)

(68, 'AF_L_model_centroids.npy')

In [6]:
import plotly.graph_objects as go

def label_from_filename(p: Path) -> str:
    # e.g. AF_L_model_centroids.npy -> AF_L
    s = p.name
    return s.replace("_model_centroids.npy", "")

def load_centroids(path: Path) -> np.ndarray:
    arr = np.load(str(path))
    if arr.ndim != 2 or arr.shape[1] != 3:
        raise ValueError(f"Unexpected centroid array shape for {path}: {arr.shape}")
    return arr.astype(float)

def make_3d_centroid_fig(points_mni: np.ndarray, label: str) -> go.Figure:
    x, y, z = points_mni[:, 0], points_mni[:, 1], points_mni[:, 2]

    fig = go.Figure()
    fig.add_trace(
        go.Scatter3d(
            x=x,
            y=y,
            z=z,
            mode="lines+markers",
            line=dict(width=2, color="rgba(30,144,255,0.35)"),
            marker=dict(size=3, color="rgba(30,144,255,0.95)"),
            name=label,
        )
    )

    fig.update_layout(
        title=label,
        scene=dict(
            xaxis_title="MNI X",
            yaxis_title="MNI Y",
            zaxis_title="MNI Z",
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        showlegend=False,
        template="plotly_white",
        height=650,
    )
    return fig


In [ ]:
if not centroid_files:
    print("No centroid files found.")
else:
    labels = [label_from_filename(p) for p in centroid_files]
    paths_by_label = {label_from_filename(p): p for p in centroid_files}

    dropdown = widgets.Dropdown(
        options=labels,
        value=labels[0],
        description="Tract:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="600px"),
    )

    out = widgets.Output()

    def _update(*_):
        with out:
            out.clear_output(wait=True)
            label = dropdown.value
            path = paths_by_label[label]
            pts = load_centroids(path)
            fig = make_3d_centroid_fig(pts, label)
            fig.show()

    dropdown.observe(_update, names="value")
    display(dropdown, out)
    _update()


## L/R pair centroids in the same 3D plot

Plots both tract centroids (e.g. `AF_L` and `AF_R`) in one figure, using the **same node index -> color mapping** for nodes 1..100.

In [1]:
def base_from_label(lbl: str) -> str:
    if lbl.endswith("_L") or lbl.endswith("_R"):
        return lbl[:-2]
    return lbl


pair_bases: list[str] = []
for lbl in list(paths_by_label.keys()):
    if lbl.endswith("_L"):
        b = base_from_label(lbl)
        if (b + "_R") in paths_by_label:
            pair_bases.append(b)

pair_bases = sorted(set(pair_bases))

if not pair_bases:
    print("No _L/_R centroid pairs found in CENTROIDS_DIR.")
else:
    dropdown = widgets.Dropdown(
        options=pair_bases,
        value=pair_bases[0],
        description="Tract base:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="600px"),
    )
    out = widgets.Output()

    def make_3d_pair_fig(points_L: np.ndarray, points_R: np.ndarray, base: str) -> go.Figure:
        xL, yL, zL = points_L[:, 0], points_L[:, 1], points_L[:, 2]
        xR, yR, zR = points_R[:, 0], points_R[:, 1], points_R[:, 2]

        # node index coloring (same for L and R)
        n = min(points_L.shape[0], points_R.shape[0])
        node_idx = np.arange(1, n + 1)

        fig = go.Figure()

        def _trace(x, y, z, name: str, line_color: str):
            return go.Scatter3d(
                x=x[:n],
                y=y[:n],
                z=z[:n],
                mode="lines+markers",
                line=dict(width=2, color=line_color),
                marker=dict(
                    size=3,
                    color=node_idx,
                    colorscale="Viridis",
                    cmin=1,
                    cmax=100,
                    showscale=False,
                    opacity=0.95,
                ),
                name=name,
            )

        fig.add_trace(_trace(xL, yL, zL, f"{base}_L", "#1f77b4"))
        fig.add_trace(_trace(xR, yR, zR, f"{base}_R", "#ff7f0e"))

        fig.update_layout(
            title=f"{base} centroids (L/R pair; node colors shared)",
            scene=dict(
                xaxis_title="MNI X",
                yaxis_title="MNI Y",
                zaxis_title="MNI Z",
                aspectmode="data",
            ),
            margin=dict(l=0, r=0, b=0, t=50),
            showlegend=True,
            template="plotly_white",
            height=650,
        )
        return fig

    def _update(*_):
        with out:
            out.clear_output(wait=True)
            base = dropdown.value
            pts_L = load_centroids(paths_by_label[base + "_L"])
            pts_R = load_centroids(paths_by_label[base + "_R"])
            fig = make_3d_pair_fig(pts_L, pts_R, base)
            fig.show()

    dropdown.observe(_update, names="value")
    display(dropdown, out)
    _update()

NameError: name 'paths_by_label' is not defined